<a href="https://colab.research.google.com/github/vikramvundyala/python_AI-ML/blob/main/U4W20_66_Hf_Fine_Tunning_BERT_Custom_Data_Set_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Certification in AIML
## A Program by IIIT-H and TalentSprint

### **Huggingface: Fine Tunning BERT using Custom Data Set**



**Learning Objectives**

At the end of the experiment you will be able to understand  :

* How to load & convert any custom dataset into a HF datasets
* Trainer and trainingArguments objects
* Computing Metrics
* Saving and loading the trained model

### Setup Steps:

In [ ]:
#@title Please enter your registration id to start: { run: "auto", display-mode: "form" }
Id = "" #@param {type:"string"}

In [ ]:
#@title Please enter your password (normally your phone number) to continue: { run: "auto", display-mode: "form" }
password = "" #@param {type:"string"}

In [ ]:
#@title Run this cell to complete the setup for this Notebook
from IPython import get_ipython
import re
ipython = get_ipython()

notebook= "U4W20_66_Hf_Fine_Tunning_BERT_Custom_Data_Set_C" #name of the notebook

def setup():
#  ipython.magic("sx pip3 install torch")
    from IPython.display import HTML, display
    display(HTML('<script src="https://dashboard.talentsprint.com/aiml/record_ip.html?traineeId={0}&recordId={1}"></script>'.format(getId(),submission_id)))
    print("Setup completed successfully")
    return

def submit_notebook():
    ipython.magic("notebook -e "+ notebook + ".ipynb")

    import requests, json, base64, datetime

    url = "https://dashboard.talentsprint.com/xp/app/save_notebook_attempts"
    if not submission_id:
      data = {"id" : getId(), "notebook" : notebook, "mobile" : getPassword()}
      r = requests.post(url, data = data)
      r = json.loads(r.text)

      if r["status"] == "Success":
          return r["record_id"]
      elif "err" in r:
        print(r["err"])
        return None
      else:
        print ("Something is wrong, the notebook will not be submitted for grading")
        return None

    elif getAnswer() and getComplexity() and getAdditional() and getConcepts() and getWalkthrough() and getComments() and getMentorSupport():
      f = open(notebook + ".ipynb", "rb")
      file_hash = base64.b64encode(f.read())

      data = {"complexity" : Complexity, "additional" :Additional,
              "concepts" : Concepts, "record_id" : submission_id,
              "answer" : Answer, "id" : Id, "file_hash" : file_hash,
              "notebook" : notebook, "feedback_walkthrough":Walkthrough ,
              "feedback_experiments_input" : Comments,
              "feedback_inclass_mentor": Mentor_support}

      r = requests.post(url, data = data)
      r = json.loads(r.text)
      if "err" in r:
        print(r["err"])
        return None
      else:
        print("Your submission is successful.")
        print("Ref Id:", submission_id)
        print("Date of submission: ", r["date"])
        print("Time of submission: ", r["time"])
        print("View your submissions: https://learn-iiith.talentsprint.com/notebook_submissions")
        #print("For any queries/discrepancies, please connect with mentors through the chat icon in LMS dashboard.")
        return submission_id
    else: submission_id


def getAdditional():
  try:
    if not Additional:
      raise NameError
    else:
      return Additional
  except NameError:
    print ("Please answer Additional Question")
    return None

def getComplexity():
  try:
    if not Complexity:
      raise NameError
    else:
      return Complexity
  except NameError:
    print ("Please answer Complexity Question")
    return None

def getConcepts():
  try:
    if not Concepts:
      raise NameError
    else:
      return Concepts
  except NameError:
    print ("Please answer Concepts Question")
    return None


def getWalkthrough():
  try:
    if not Walkthrough:
      raise NameError
    else:
      return Walkthrough
  except NameError:
    print ("Please answer Walkthrough Question")
    return None

def getComments():
  try:
    if not Comments:
      raise NameError
    else:
      return Comments
  except NameError:
    print ("Please answer Comments Question")
    return None


def getMentorSupport():
  try:
    if not Mentor_support:
      raise NameError
    else:
      return Mentor_support
  except NameError:
    print ("Please answer Mentor support Question")
    return None

def getAnswer():
  try:
    if not Answer:
      raise NameError
    else:
      return Answer
  except NameError:
    print ("Please answer Question")
    return None


def getId():
  try:
    return Id if Id else None
  except NameError:
    return None

def getPassword():
  try:
    return password if password else None
  except NameError:
    return None

submission_id = None
### Setup
if getPassword() and getId():
  submission_id = submit_notebook()
  if submission_id:
    setup()
else:
  print ("Please complete Id and Password cells before running setup")



In [ ]:
# @title Download dataset
!pip install --upgrade gdown
!gdown "1JlNQOwHcWiq7pubASjs8XaGP0feuK2_E"

## Importing packages

Accelerate is a library that enable the same Pytorch code to run accross any distributed configuration by adding just four lines of code, making training and interface at scale made simple, efficient and adaptable.

In [ ]:
!pip install accelerate -U

Huggingface dataset library

In [ ]:
!pip install transformers datasets

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import seaborn as sn
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

### **Loading the Custom Data**

In [ ]:
# Read CSV file
df_ = pd.read_csv('/content/Tweets.csv')
df_.head()

In [ ]:
# Select the respective columns to identify the sentiment of text
df = df_[['airline_sentiment', 'text']].copy()
df.head()

In [ ]:
# Plot the histogram and check the ratio of split amongst sentiment
df['airline_sentiment'].hist()

In [ ]:
# Convert the characters to numeric
target_map ={'positive':1, 'negative':0, 'neutral':2}

# Create the new column and map the changes
df['target'] = df['airline_sentiment'].map(target_map)

In [ ]:
# Assign to new object and rename the column names
df2 = df[['text','target']]
df2.columns=['sentence','label']

# Save the csv file
df2.to_csv('data.csv',index=None)

In [ ]:
!head data.csv

### **Finally conveting the data.csv into HF dataset**

In [ ]:
# load the dataset through HF transformer library dataset
from datasets import load_dataset
raw_dataset = load_dataset('csv', data_files='data.csv')

In [ ]:
raw_dataset

In [ ]:
# Split the data set into train and test
split = raw_dataset['train'].train_test_split(test_size = 0.3, seed=42)

In [ ]:
split

In [ ]:
# for multiple csv files
# load_dataset('csv', data_files =['file1.csv', 'file2.csv'])

In [ ]:
# if you already have a train-test split :
# load_dataset('csv', data_files ={'train':['train1.csv','train2.csv'],'test':'test.csv'})

### **Tokenizing**

In [ ]:
# creating checkpoint of bert model
checkpoint = 'distilbert-base-cased'

In [ ]:
from transformers import AutoTokenizer

In [ ]:
# Autotokenizer to load the tokenizer internally upon the model provided.
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

By calling the map method and passing in a tokenize function, the dataset library automatically apply the same function to every train and test set.

*Note*: In this example, we appllied only truncation but not padding or conversion into Pytorch tensor. This will be handled by the trainer object, created later.

In [ ]:

def tokenize_fn(batch):
  return tokenizer(batch['sentence'],truncation = True)

In [ ]:
# Map a python fuinction will operate accross the every batch
tokenized_datasets = split.map(tokenize_fn,  batched=True)

### **Defining Model**

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

*Note:* The checkpoint we pass here in must match the checkpoint we passed in tokenizer. so, we get right tokenizers for the model.

In [ ]:
# AutoModelForSequenceClassification is used to get a text classification model from the checkpoint.
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=3)

In [ ]:
# To display summary we install torchinfo
!pip install torchinfo

In [ ]:
from torchinfo import summary

In [ ]:
summary(model)

### **Training Arguments**

TrainingArguments is the subset of the arguments we use in our example scripts **which relate to the training loop
itself**.

In [ ]:
training_args = TrainingArguments(
    output_dir = 'training_dir', # The output directory where the model predictions and checkpoints will be written.
    eval_strategy='epoch', # Evaluation is done at the end of each epoch.
    save_strategy='epoch', # Save is done at the end of each epoch.
    num_train_epochs=3, # Total number of training epochs to perform.
    per_device_train_batch_size =16, # The batch size for training.
    per_device_eval_batch_size = 64, # The batch size for Evaluation.
)

### **Metrics**

In [ ]:
def compute_metrics(logits_and_labels):
  logits, labels = logits_and_labels
  predictions =np.argmax(logits,axis=-1)
  acc=np.mean(predictions == labels) # return accruacy
  f1=f1_score(labels,predictions, average='macro') # return F1 score
  return {'accuracy':acc, 'f1':f1} # returns as a dictionary

### **Training**

In [ ]:
trainer = Trainer(
    model, # pre-trained model
    training_args, # training arguments
    train_dataset=tokenized_datasets["train"], # the data use for training
    eval_dataset = tokenized_datasets["test"], # the data use for evaluation
    tokenizer = tokenizer, # The tokenizer to preprocess the data.
    compute_metrics=compute_metrics # The function that will be used to compute metrics at evaluation
)

In [ ]:
trainer.train()

### **Loading the saved model**

In [ ]:
# Save has done at the end of each epoch in training_dir
!ls training_dir

In [ ]:
from transformers import pipeline

In [ ]:
# method to build a Pipeline, load the saved model from the above checkpoints as a final model, device on which this pipeline will be allocated.
my_model = pipeline('text-classification',model='training_dir/checkpoint-641',device=0)

In [ ]:
split['test']

### **Testing**

In [ ]:
# Pass the test sentence data to model
test_pred = my_model(split['test']['sentence'])

In [ ]:
# get the label from the pridictions
def get_label(d):
  return int(d['label'].split('_')[1])

# Append it to the test_pred
test_pred =[get_label(d) for d in test_pred]

In [ ]:
print("accuracy : ", accuracy_score(split['test']['label'], test_pred))

### Please answer the questions below to complete the experiment:




In [ ]:
#@title Select True or False: In the context of the BERT model that we have implemented, tokenization is just word-level separation from sentences? { run: "auto", form-width: "500px", display-mode: "form" }
Answer = "" #@param ["","TRUE", "FALSE"]

In [ ]:
#@title How was the experiment? { run: "auto", form-width: "500px", display-mode: "form" }
Complexity = "" #@param ["","Too Simple, I am wasting time", "Good, But Not Challenging for me", "Good and Challenging for me", "Was Tough, but I did it", "Too Difficult for me"]


In [ ]:
#@title If it was too easy, what more would you have liked to be added? If it was very difficult, what would you have liked to have been removed? { run: "auto", display-mode: "form" }
Additional = "" #@param {type:"string"}


In [ ]:
#@title Can you identify the concepts from the lecture which this experiment covered? { run: "auto", vertical-output: true, display-mode: "form" }
Concepts = "" #@param ["","Yes", "No"]


In [ ]:
#@title  Experiment walkthrough video? { run: "auto", vertical-output: true, display-mode: "form" }
Walkthrough = "" #@param ["","Very Useful", "Somewhat Useful", "Not Useful", "Didn't use"]


In [ ]:
#@title  Text and image description/explanation and code comments within the experiment: { run: "auto", vertical-output: true, display-mode: "form" }
Comments = "" #@param ["","Very Useful", "Somewhat Useful", "Not Useful", "Didn't use"]


In [ ]:
#@title Mentor Support: { run: "auto", vertical-output: true, display-mode: "form" }
Mentor_support = "" #@param ["","Very Useful", "Somewhat Useful", "Not Useful", "Didn't use"]


In [ ]:
#@title Run this cell to submit your notebook for grading { vertical-output: true }
try:
  if submission_id:
      return_id = submit_notebook()
      if return_id : submission_id = return_id
  else:
      print("Please complete the setup first.")
except NameError:
  print ("Please complete the setup first.")